# MiniMax H3 · Motion Context · Colab Remote

此 notebook 包含全部執行程式，單獨上傳 `.ipynb` 即可。
先選擇 GPU runtime，建議使用高 RAM、大 VRAM，並確認模型所需磁碟空間。

**手動生成：Cell 1 → 2 → 3 → 4 → 5 → 6。遠端 AI：Cell 1 → 2 → 3 → 4 → 5 → 7。**
Cell 3 改參數後按「套用全部設定」；換模型／LoRA 後執行 Cell 4。

流程：模型 → 有序 LoRA → Sigma Shift → Kitchen → 可選顯存分塊 → Sol → T8 conditioning →
可選 Motion Context → 一次完整採樣 → 影像與音訊解碼 → 同步裁掉 context → MP4。

| 項目 | 預設 |
|---|---|
| Kitchen / Sol attention | 開啟 / 開啟 |
| 降低顯存 | 關閉；可調 head_chunks、ff_chunks、seq_threshold |
| Motion Context | 關閉，開啟時必須選前段影片 |
| context / audio context | 22 / 24 影格 |
| 生成 | 1152×640，10 秒，euler / beta，8 steps，隨機 seed |

採樣秒數會對齊 H3 的 17n+5 影格網格（24fps）。啟用 context 時，重疊影格會從成片開頭裁掉，
因此實際新片長較短。只輸出新片段，不自動合併前段。可選任何 MP4 與其中一段區間；用該區間尾端接續。
首段關閉串接即可。前段不是 `<Video N>` 或 `<Audio N>` 參考素材。
Motion Context 佔用開頭，串接模式用 Ref2VA、T2VA 或 L2VA。


## Cell 1 — 安裝 ComfyUI 與節點
安裝 T8、VideoHelperSuite、H3 Motion Context、KJNodes，記錄 commits。可在本格設定 REVISIONS 固定版本。

In [ ]:
"""Cell 1: install runtime dependencies in the temporary Colab /content folder."""
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

assert torch.cuda.is_available(), '請在 Colab 選擇 GPU 執行階段'
COMFY = Path('/content/ComfyUI')
REPOS = {
    'ComfyUI': ('https://github.com/Comfy-Org/ComfyUI.git', COMFY),
    'T8': ('https://github.com/T8mars/comfyui-minimax-h3-audio-T8.git', COMFY/'custom_nodes/comfyui-minimax-h3-audio-T8'),
    'VHS': ('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git', COMFY/'custom_nodes/ComfyUI-VideoHelperSuite'),
    'MotionContext': ('https://github.com/NikoDemon80/ComfyUI-H3-Motion-Context.git', COMFY/'custom_nodes/ComfyUI-H3-Motion-Context'),
    'KJNodes': ('https://github.com/kijai/ComfyUI-KJNodes.git', COMFY/'custom_nodes/ComfyUI-KJNodes'),
}
# Optional reproducibility: put exact revisions here before running this cell.
REVISIONS = {}  # e.g. {'ComfyUI': '<commit>', 'MotionContext': '<commit>'}
if globals().get('ENGINE') and ENGINE.queue.unfinished_tasks:
    raise RuntimeError('有未完成工作，請等待後再安裝')
if globals().get('COMFY_PROCESS') and COMFY_PROCESS.poll() is None:
    raise RuntimeError('請先重新啟動 Colab 執行階段，再安裝節點')
PROVENANCE = {}
for name, (url, dest) in REPOS.items():
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--depth', '1', url, str(dest)], check=True)
    if not (dest/'.git').is_dir():
        raise RuntimeError(f'{dest} 不是 Git clone')
    revision = REVISIONS.get(name, 'HEAD')
    subprocess.run(['git', '-C', str(dest), 'fetch', '--depth', '1', 'origin', revision], check=True)
    # Detached checkout keeps temporary clones reproducible without rewriting user projects.
    subprocess.run(['git', '-C', str(dest), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    requirements = dest/'requirements.txt'
    if requirements.is_file():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
    PROVENANCE[name] = subprocess.check_output(['git', '-C', str(dest), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'hf_xet',
                'ipywidgets', 'pillow', 'requests', 'fastapi', 'uvicorn', 'python-multipart', 'httpx'], check=True)
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), '需要 ffmpeg / ffprobe'
for sub in ('input', 'output'):
    (COMFY/sub).mkdir(exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))
print('Free disk GiB:', round(shutil.disk_usage('/content').free/2**30, 2))
print('Repository commits:', json.dumps(PROVENANCE, indent=2))
print('節點已安裝，接著執行設定格。')


## Cell 2 — 載入生成引擎
不會執行 GPU 工作。所有手動／API 生成共用此程式。

In [ ]:
"""MiniMax H3 Motion Context: shared manual/API generation engine (Python 3.10+)."""
import asyncio
import copy
import json
import math
import os
import queue
import re
import secrets
import shutil
import subprocess
import threading
import time
import uuid
from pathlib import Path, PureWindowsPath

import requests

PROTOCOL = 'h3-motion-context-1'
DEFAULTS = {
    'mode': 'ref2va', 'duration': 10.0, 'width': 1152, 'height': 640,
    'seed': '', 'steps': 8, 'sampler': 'euler', 'scheduler': 'beta',
    'model': 'minimax_h3_ref2va_pruned_int8_convrot.safetensors',
    'text_encoder': 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
    'video_vae': 'minimax_h3_video_vae_int8_convrot.safetensors',
    'audio_vae': 'minimax_h3_audio_vae_fp32.safetensors',
    'loras': [], 'shift_enabled': True, 'shift_video': 6.0, 'shift_audio': 3.0,
    'kitchen_attention': True, 'sol_attention': True, 'sol_tau': 1.3,
    'sol_start': 0.2, 'sol_end': 1.0, 'sol_dense_blocks': '',
    'sol_min_tokens': 12288, 'sol_extra_tokens': 256, 'sol_verbose': True,
    'low_vram': False, 'head_chunks': 4, 'ff_chunks': 2, 'seq_threshold': 4096,
    'motion_context': False, 'context_length': 22, 'audio_context_length': 24,
    'previous_start': 0.0, 'previous_end': None,
    'ref_image_size': 'match', 'reference_video_policy': 'official_2_to_15s',
    'crf': 19, 'prefix': 'H3',
}


def safe_name(value):
    value = str(value).replace('\\', '/')
    if not value or value.startswith('/') or PureWindowsPath(value).drive or '..' in value.split('/'):
        raise ValueError('模型名稱必須是模型目錄內的相對路徑')
    return value


def normalize(overrides=None, defaults=None):
    cfg = copy.deepcopy(DEFAULTS if defaults is None else defaults)
    overrides = overrides or {}
    unknown = set(overrides) - set(DEFAULTS)
    if unknown:
        raise ValueError(f'未知設定: {sorted(unknown)}')
    cfg.update(copy.deepcopy(overrides))
    bools = [k for k, v in DEFAULTS.items() if type(v) is bool]
    for k in bools:
        if type(cfg[k]) is not bool:
            raise ValueError(f'{k} 必須是 boolean')
    if not cfg['motion_context']:
        cfg['previous_start'], cfg['previous_end'] = 0.0, None
    ints = {'width': (32, 16384), 'height': (32, 16384), 'steps': (1, 10000),
            'head_chunks': (1, 56), 'ff_chunks': (1, 64), 'seq_threshold': (256, 262144),
            'context_length': (5, 56), 'audio_context_length': (0, 240),
            'sol_min_tokens': (0, 1048576), 'sol_extra_tokens': (0, 256), 'crf': (0, 51)}
    for k, (lo, hi) in ints.items():
        if type(cfg[k]) is not int or not lo <= cfg[k] <= hi:
            raise ValueError(f'{k} 必須是 {lo}–{hi} 整數')
    for k, lo, hi in [('duration', 0.21, 600), ('shift_video', 0.01, 100),
                       ('shift_audio', 0.01, 100), ('sol_tau', 0, 4),
                       ('sol_start', 0, 1), ('sol_end', 0, 1), ('previous_start', 0, 86400)]:
        if type(cfg[k]) not in (float, int) or not math.isfinite(cfg[k]) or not lo <= cfg[k] <= hi:
            raise ValueError(f'{k} 必須是 {lo}–{hi} 有限數值')
    if cfg['width'] % 32 or cfg['height'] % 32:
        raise ValueError('寬高必須是 32 的倍數')
    if cfg['context_length'] not in (5, 22, 39, 56):
        raise ValueError('context_length: 5 / 22 / 39 / 56')
    if cfg['motion_context'] and frame_count(cfg['duration']) <= cfg['context_length']:
        raise ValueError('採樣影格數必須大於 context_length')
    if cfg['sol_start'] > cfg['sol_end']:
        raise ValueError('sol_start 必須 <= sol_end')
    if cfg['mode'] not in ('ref2va', 't2va', 'i2va', 'fl2va', 'l2va'):
        raise ValueError('mode: ref2va / t2va / i2va / fl2va / l2va')
    if cfg['ref_image_size'] not in ('match', 'max') or cfg['reference_video_policy'] not in ('official_2_to_15s', 'model_minimum'):
        raise ValueError('參考素材處理選項不合法')
    end = cfg['previous_end']
    if end is not None and (type(end) not in (float, int) or not math.isfinite(end) or end <= cfg['previous_start']):
        raise ValueError('previous_end 必須大於 previous_start，或為 null')
    seed = str(cfg['seed']).strip()
    if seed and (not re.fullmatch(r'\d+', seed) or int(seed) > 2**64-1):
        raise ValueError('seed 必須是 0–18446744073709551615 的十進位字串，空白為隨機')
    cfg['seed'] = str(secrets.randbits(64)) if not seed else seed
    for k in ('model', 'text_encoder', 'video_vae', 'audio_vae'):
        cfg[k] = safe_name(cfg[k])
    if not isinstance(cfg['loras'], list):
        raise ValueError('loras 必須是依套用順序排列的陣列')
    for entry in cfg['loras']:
        if not isinstance(entry, dict) or set(entry) != {'name', 'strength'}:
            raise ValueError('每份 LoRA 必須是 {name, strength}')
        entry['name'] = safe_name(entry['name'])
        if type(entry['strength']) not in (float, int) or not math.isfinite(entry['strength']):
            raise ValueError('LoRA strength 必須為有限數值')
    if not isinstance(cfg['prefix'], str) or not re.fullmatch(r'[\w-]{1,80}', cfg['prefix']):
        raise ValueError('prefix 只接受 1–80 個文字、數字、底線或連字號')
    return cfg


def frame_count(seconds):
    # Match the supplied ComfyMathExpression, including Python round().
    n = max(5, round(seconds * 24))
    return n + (5 - n % 17) % 17


def probe(path):
    p = subprocess.run(['ffprobe', '-v', 'error', '-show_streams', '-show_format',
                        '-of', 'json', str(path)], capture_output=True, text=True, check=True)
    return json.loads(p.stdout)


def has_audio(path):
    return any(s.get('codec_type') == 'audio' for s in probe(path)['streams'])


def validate_media(cfg, media, prompt):
    counts = {k: len(media.get(k, [])) for k in ('images', 'videos', 'audios')}
    n, v, a = (counts[k] for k in ('images', 'videos', 'audios'))
    if n > 9 or v > 3 or a > 3:
        raise ValueError('最多 9 張 Picture、3 個 Video、3 個獨立 Audio')
    mode = cfg['mode']
    if mode == 'ref2va' and not (n or v or a):
        raise ValueError('Ref2VA 至少需要一份參考素材；前段影片不算參考素材，可改用 T2VA')
    if mode != 'ref2va' and (v or a):
        raise ValueError('Video / Audio 參考素材僅供 Ref2VA')
    if mode != 'ref2va' and n != {'t2va': 0, 'i2va': 1, 'fl2va': 2, 'l2va': 1}[mode]:
        raise ValueError('T2VA=0、I2VA=1 首幀、FL2VA=首幀+尾幀、L2VA=1 尾幀')
    if cfg['motion_context'] != bool(media.get('previous')):
        raise ValueError('motion_context=true 必須提供 previous；關閉時請移除 previous')
    if cfg['motion_context'] and mode in ('i2va', 'fl2va'):
        raise ValueError('Motion Context 擁有開頭影格，請用 Ref2VA / T2VA / L2VA')
    for kind, index in re.findall(r'<(Picture|Video|Audio)\s+(\d+)>', prompt):
        if mode != 'ref2va' or not 1 <= int(index) <= counts[{'Picture': 'images', 'Video': 'videos', 'Audio': 'audios'}[kind]]:
            raise ValueError(f'<{kind} {index}> 沒有對應素材')
    if not prompt.strip():
        raise ValueError('prompt 不可為空')


def schema_fields(spec, inputs):
    fields = dict(spec.get('required', {}), **spec.get('optional', {}))
    required = set(spec.get('required', {}))
    for key, val in list(fields.items()):
        meta = val[1] if len(val) > 1 and isinstance(val[1], dict) else {}
        # VHS format widgets live in metadata, not in the optional input table.
        selected_format = inputs.get(key) if isinstance(inputs.get(key), str) else ''
        for widget in meta.get('formats', {}).get(selected_format, []):
            fields[widget[0]] = widget[1:]
        if val[0] == 'COMFY_DYNAMICCOMBO_V3':
            for opt in meta.get('options', []):
                if opt['key'] == inputs.get(key):
                    for group in ('required', 'optional'):
                        for child, child_spec in opt['inputs'].get(group, {}).items():
                            fields[f'{key}.{child}'] = child_spec
                            if group == 'required':
                                required.add(f'{key}.{child}')
        if val[0] == 'COMFY_AUTOGROW_V3':
            template = meta.get('template', {})
            child = template.get('input', {})
            child_spec = next(iter(dict(child.get('required', {}), **child.get('optional', {})).values()), None)
            names = template.get('names') or [template.get('prefix', '') + str(i) for i in range(template.get('max', 0))]
            for name in names:
                fields[f'{key}.{name}'] = child_spec
            required.discard(key)
    return fields, required


def validate_graph(graph, info):
    for ident, node in graph.items():
        kind, inputs = node['class_type'], node['inputs']
        if kind not in info:
            raise ValueError(f'缺少 {kind}，請重跑安裝與啟動格，查看 ComfyUI log')
        fields, required = schema_fields(info[kind]['input'], inputs)
        if set(inputs) - set(fields) or required - set(inputs):
            raise ValueError(f'{kind} schema 不符: unknown={set(inputs)-set(fields)}, missing={required-set(inputs)}')
        for key, value in inputs.items():
            field = fields[key]
            if isinstance(value, list) and len(value) == 2 and isinstance(value[0], str) and type(value[1]) is int:
                if value[0] not in graph:
                    raise ValueError(f'{ident}.{key} 無效連線')
                outputs = info[graph[value[0]]['class_type']]['output']
                if not 0 <= value[1] < len(outputs):
                    raise ValueError(f'{ident}.{key} 輸出索引無效')
                expected = field[0] if field else '*'
                if isinstance(expected, str) and expected != '*' and outputs[value[1]] != '*' and not set(str(outputs[value[1]]).split(',')) & set(expected.split(',')):
                    raise ValueError(f'{kind}.{key}: {outputs[value[1]]} != {expected}')
                continue
            if not field:
                continue
            meta = field[1] if len(field) > 1 and isinstance(field[1], dict) else {}
            options = field[0] if isinstance(field[0], list) else None
            if field[0] == 'COMFY_DYNAMICCOMBO_V3':
                options = [x['key'] for x in meta.get('options', [])]
            if options is not None and value not in options and kind not in ('LoadImage', 'LoadAudio', 'VHS_LoadVideo'):
                raise ValueError(f'{kind}.{key} 不在安裝清單: {value}')
            if type(value) in (float, int) and (not math.isfinite(value) or value < meta.get('min', -math.inf) or value > meta.get('max', math.inf)):
                raise ValueError(f'{kind}.{key} 超出節點範圍')


def build_graph(cfg, media, prompt, info):
    validate_media(cfg, media, prompt)
    graph = {}
    def add(kind, **inputs):
        ident = str(len(graph) + 1)
        graph[ident] = {'class_type': kind, 'inputs': inputs}
        return [ident, 0]
    model = add('UNETLoader', unet_name=cfg['model'], weight_dtype='default')
    clip = add('CLIPLoader', clip_name=cfg['text_encoder'], type='minimax', device='default')
    vv = add('VAELoader', vae_name=cfg['video_vae'])
    av = add('VAELoader', vae_name=cfg['audio_vae'])
    for lora in cfg['loras']:
        model = add('LoraLoaderModelOnly', model=model, lora_name=lora['name'], strength_model=lora['strength'])
    if cfg['shift_enabled']:
        model = add('MiniMaxH3SigmaShift', model=model, shift_video=cfg['shift_video'], shift_audio=cfg['shift_audio'])
    if cfg['kitchen_attention']:
        model = add('ModelAttentionBackend', model=model, attention='comfy kitchen attention')
    if cfg['low_vram']:
        model = add('MiniMaxLowVRAMAttention', model=model, head_chunks=cfg['head_chunks'])
        model = add('MiniMaxChunkFeedForward', model=model, chunks=cfg['ff_chunks'], seq_threshold=cfg['seq_threshold'])
    if cfg['sol_attention']:
        model = add('BlockSparseAttention', model=model, selection='sol-attn',
                    **{'selection.tau': cfg['sol_tau']}, start_percent=cfg['sol_start'], end_percent=cfg['sol_end'],
                    dense_blocks=cfg['sol_dense_blocks'], min_tokens=cfg['sol_min_tokens'],
                    extra_tokens=cfg['sol_extra_tokens'], sink_conditioning='exact_kv_and_rows', verbose=cfg['sol_verbose'])
    tasks = {'ref2va': 'Ref2VA', 't2va': 'T2VA', 'i2va': 'I2VA', 'fl2va': 'FL2VA', 'l2va': 'L2VA'}
    task = tasks[cfg['mode']]
    options = info.get('MiniMaxH3AudioConditioningT8', {}).get('input', {}).get('required', {}).get('task_type', [[]])[0]
    if isinstance(options, list):
        task = next((x for x in options if x == task or x.startswith(task + ' ')), task)
    base = dict(clip=clip, video_vae=vv, audio_vae=av, prompt=prompt, width=cfg['width'], height=cfg['height'],
                length=frame_count(cfg['duration']), task_type=task, audio_mode='native', audio_denoise_strength=1.0,
                add_source_as_reference=False, prompt_primary_audio_ordinal=0, strict_prompt_tags=True,
                ref_image_size=cfg['ref_image_size'], reference_video_policy=cfg['reference_video_policy'])
    def video(name):
        return add('VHS_LoadVideo', video=name, force_rate=24, custom_width=0, custom_height=0,
                   frame_load_cap=0, skip_first_frames=0, select_every_nth=1)
    if cfg['mode'] == 'ref2va':
        for i, name in enumerate(media.get('images', [])):
            base[f'ref_images.ref_image_{i}'] = add('LoadImage', image=name)
        offset = 0
        for i, name in enumerate(media.get('videos', [])):
            load = video(name)
            base[f'ref_videos.ref_video_{i}'] = load
            if media.get('video_has_audio', [False] * len(media['videos']))[i]:
                base[f'ref_video_audios.ref_video_audio_{i}'] = [load[0], 2]
                offset += 1
        for i, name in enumerate(media.get('audios', [])):
            base[f'ref_audios.ref_audio_{i}'] = add('LoadAudio', audio=name)
        # Public Audio N always means standalone Audio N; T8 puts video soundtracks first.
        base['prompt'] = re.sub(r'<Audio\s+(\d+)>', lambda m: f'<Audio {int(m[1])+offset}>', prompt)
    elif cfg['mode'] in ('i2va', 'fl2va'):
        base['first_frame'] = add('LoadImage', image=media['images'][0])
        if cfg['mode'] == 'fl2va':
            base['last_frame'] = add('LoadImage', image=media['images'][1])
    elif cfg['mode'] == 'l2va':
        base['last_frame'] = add('LoadImage', image=media['images'][0])
    cond = add('MiniMaxH3AudioConditioningT8', **base)
    latent = [cond[0], 1]
    trim = 0
    if cfg['motion_context']:
        prev = video(media['previous'])
        mc_inputs = dict(conditioning=cond, vae=vv, latent=latent, context_frames=prev,
                         context_length=str(cfg['context_length']), audio_context_length=cfg['audio_context_length'])
        if media.get('previous_has_audio'):
            mc_inputs.update(audio_vae=av, context_audio=[prev[0], 2])
        cond = add('MiniMaxH3MotionContext', **mc_inputs)
        trim = [cond[0], 1]
    guider = add('BasicGuider', model=model, conditioning=cond)
    noise = add('RandomNoise', noise_seed=int(cfg['seed']))
    sampler = add('KSamplerSelect', sampler_name=cfg['sampler'])
    sigmas = add('BasicScheduler', model=model, scheduler=cfg['scheduler'], steps=cfg['steps'], denoise=1.0)
    sampled = add('SamplerCustomAdvanced', noise=noise, guider=guider, sampler=sampler, sigmas=sigmas, latent_image=latent)
    frames = add('VAEDecode', samples=sampled, vae=vv)
    audio = add('VAEDecodeAudio', samples=sampled, vae=av)
    trimmed = add('MiniMaxH3MotionContextTrim', images=frames, audio=audio, trim_frames=trim, fps=24.0, match_tail=True)
    output = add('VHS_VideoCombine', images=trimmed, audio=[trimmed[0], 1], frame_rate=24.0, loop_count=0,
                 filename_prefix=cfg['output_prefix'], format='video/h264-mp4', pix_fmt='yuv420p', crf=cfg['crf'],
                 save_metadata=True, trim_to_audio=False, pingpong=False, save_output=True)
    validate_graph(graph, info)
    return graph, output[0], base['prompt']


class Engine:
    def __init__(self, comfy='/content/ComfyUI', url='http://127.0.0.1:8188', defaults=None, provenance=None):
        self.comfy = Path(comfy)
        self.url = url.rstrip('/')
        self.defaults = copy.deepcopy(defaults or DEFAULTS)
        self.provenance = provenance or {}
        self.root = self.comfy / 'output' / 'h3_remote'
        self.root.mkdir(parents=True, exist_ok=True)
        self.jobs = {}
        self.lock = threading.RLock()
        self.queue = queue.Queue()
        self.thread = threading.Thread(target=self._worker, daemon=True)
        self.thread.start()

    def info(self):
        r = requests.get(self.url + '/object_info', timeout=60)
        r.raise_for_status()
        return r.json()

    def status(self, ident):
        with self.lock:
            if ident not in self.jobs:
                raise KeyError(ident)
            return copy.deepcopy(self.jobs[ident])

    def update(self, ident, **changes):
        with self.lock:
            self.jobs[ident].update(changes)

    def submit(self, cfg, media, prompt):
        cfg = normalize(cfg, self.defaults)
        validate_media(cfg, media, prompt)
        ident = uuid.uuid4().hex
        cfg['output_prefix'] = f'h3_remote/{ident}/{cfg["prefix"]}_seed{cfg["seed"]}'
        # Check node/model contracts before accepting a GPU job.
        self.enrich(media)
        graph, output, conditioning_prompt = build_graph(cfg, media, prompt, self.info())
        self._save_json(self.root / ident / 'workflow.json', graph)
        with self.lock:
            self.jobs[ident] = {'id': ident, 'status': 'queued', 'seed': cfg['seed'], 'submitted': time.time()}
        self.queue.put((ident, cfg, copy.deepcopy(media), prompt, graph, output, conditioning_prompt))
        return ident

    @staticmethod
    def _save_json(path, data):
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')

    def enrich(self, media):
        media['video_has_audio'] = [has_audio(self.comfy / 'input' / p) for p in media.get('videos', [])]
        if media.get('previous'):
            media['previous_has_audio'] = has_audio(self.comfy / 'input' / media['previous'])

    def prepare_previous(self, cfg, media, directory):
        if not cfg['motion_context']:
            return
        source = self.comfy / 'input' / media['previous']
        data = probe(source)
        if not any(s.get('codec_type') == 'video' for s in data['streams']):
            raise ValueError('前段檔案沒有影像串流')
        duration = float(data['format']['duration'])
        end = cfg['previous_end'] if cfg['previous_end'] is not None else duration
        start = cfg['previous_start']
        if end > duration + 0.05 or end <= start:
            raise ValueError(f'前段選取區間超出影片長度 {duration:.3f}s')
        # Decode only the tail needed by Motion Context, on an exact 24fps grid.
        # Keep enough audio when its lookback exceeds the visual lookback.
        window = max(cfg['context_length'], cfg['audio_context_length']) / 24 + 0.5
        tail_start = max(start, end - window)
        dest = directory / 'previous_tail.mp4'
        command = ['ffmpeg', '-v', 'error', '-y', '-ss', str(tail_start), '-i', str(source),
                   '-t', str(end-tail_start), '-map', '0:v:0', '-map', '0:a:0?',
                   '-vf', 'fps=24', '-c:v', 'libx264', '-crf', '16', '-pix_fmt', 'yuv420p',
                   '-c:a', 'aac', '-ar', '48000', str(dest)]
        subprocess.run(command, check=True)
        details = probe(dest)
        stream = next(s for s in details['streams'] if s['codec_type'] == 'video')
        if int(stream.get('nb_frames', 0)) < cfg['context_length']:
            raise ValueError('所選前段區間太短，沒有足夠 context_length 影格')
        media['previous_original'] = media['previous']
        media['previous'] = dest.relative_to(self.comfy / 'input').as_posix()
        media['previous_has_audio'] = any(s['codec_type'] == 'audio' for s in details['streams'])
        media['previous_selected_range'] = [start, end]
        media['previous_tail_start'] = tail_start

    def _worker(self):
        while True:
            item = self.queue.get()
            try:
                if item is None: return
                self._run(*item)
            except Exception as exc:
                self.update(item[0], status='failed', error=str(exc))
            finally:
                self.queue.task_done()

    def _run(self, ident, cfg, media, prompt, graph, output, conditioning_prompt):
        self.update(ident, status='preparing')
        directory = self.comfy / 'input' / 'h3_remote' / ident
        directory.mkdir(parents=True, exist_ok=True)
        self.prepare_previous(cfg, media, directory)
        graph, output, conditioning_prompt = build_graph(cfg, media, prompt, self.info())
        self._save_json(self.root / ident / 'workflow.json', graph)
        try:
            r = requests.post(self.url + '/prompt', json={'prompt': graph, 'client_id': ident}, timeout=60)
        except requests.RequestException:
            self.update(ident, status='submission_unknown', error='提交連線中斷，ComfyUI 可能已接收；請查看 Colab queue/log，勿重送。引擎暫停後續工作。')
            # Hold the serial queue: accepting another GPU task would hide an uncertain submission.
            threading.Event().wait()
            return
        if not r.ok:
            raise RuntimeError(f'ComfyUI rejected graph: {r.status_code}: {r.text}')
        result = r.json()
        if result.get('node_errors') or not result.get('prompt_id'):
            raise RuntimeError(json.dumps(result, ensure_ascii=False))
        prompt_id = result['prompt_id']
        self.update(ident, status='running', prompt_id=prompt_id, started=time.time())
        while True:
            try:
                r = requests.get(self.url + '/history/' + prompt_id, timeout=30)
                r.raise_for_status()
                history = r.json().get(prompt_id)
            except requests.RequestException:
                # A network timeout is not evidence that the GPU job stopped.
                time.sleep(5)
                continue
            if history:
                status = history.get('status', {})
                if status.get('status_str') == 'error':
                    raise RuntimeError(json.dumps(status, ensure_ascii=False))
                if status.get('completed') or status.get('status_str') == 'success':
                    break
            time.sleep(3)
        candidates = []
        def walk(obj):
            if isinstance(obj, dict):
                if str(obj.get('filename', '')).lower().endswith('.mp4') and obj.get('type') == 'output':
                    candidates.append(obj)
                for v in obj.values(): walk(v)
            elif isinstance(obj, list):
                for v in obj: walk(v)
        walk(history.get('outputs', {}).get(output, {}))
        paths = [(self.comfy / 'output' / x.get('subfolder', '') / x['filename']).resolve() for x in candidates]
        paths = [p for p in paths if p.is_relative_to(self.root.resolve()) and p.is_file() and p.stat().st_size]
        if not paths:
            raise RuntimeError('ComfyUI 完成，但輸出節點未回報 MP4')
        video = paths[-1]
        data = probe(video)
        kinds = {s['codec_type'] for s in data['streams']}
        if not {'video', 'audio'} <= kinds:
            raise RuntimeError('MP4 缺少影像或音訊串流')
        v = next(s for s in data['streams'] if s['codec_type'] == 'video')
        # Record real output properties; the context head reduces delivered duration.
        record = {'protocol': PROTOCOL, 'job_id': ident, 'prompt_id': prompt_id,
                  'settings': cfg, 'media': media, 'prompt': prompt, 'conditioning_prompt': conditioning_prompt,
                  'sampled_frames': frame_count(cfg['duration']),
                  'output': {'width': v['width'], 'height': v['height'], 'duration': data['format']['duration'],
                             'frames': v.get('nb_frames'), 'fps': v.get('avg_frame_rate')},
                  'provenance': self.provenance}
        txt = video.with_suffix('.txt')
        self._save_json(txt, record)
        self.update(ident, status='completed', video=str(video), info=str(txt), finished=time.time(), output=record['output'])


def create_app(engine, token):
    from fastapi import FastAPI, Depends, File, Form, Header, HTTPException, UploadFile
    from fastapi.responses import FileResponse
    from typing import List, Optional
    def auth(authorization: str = Header('')):
        if not secrets.compare_digest(authorization, 'Bearer ' + token):
            raise HTTPException(401, 'Unauthorized')
    app = FastAPI(title='MiniMax H3 Motion Context', dependencies=[Depends(auth)])
    @app.get('/health')
    def health():
        engine.info()
        return {'protocol': PROTOCOL, 'ready': True, 'defaults': engine.defaults, 'provenance': engine.provenance}
    @app.get('/models')
    def models():
        info = engine.info()
        fields = {'models': ('UNETLoader', 'unet_name'), 'text_encoders': ('CLIPLoader', 'clip_name'),
                  'vaes': ('VAELoader', 'vae_name'), 'loras': ('LoraLoaderModelOnly', 'lora_name'),
                  'samplers': ('KSamplerSelect', 'sampler_name'), 'schedulers': ('BasicScheduler', 'scheduler')}
        return {k: info[n]['input']['required'][f][0] for k, (n, f) in fields.items()}
    @app.post('/generate', status_code=202)
    async def generate(prompt: str = Form(...), settings: str = Form('{}'),
                       images: Optional[List[UploadFile]] = File(None),
                       videos: Optional[List[UploadFile]] = File(None),
                       audios: Optional[List[UploadFile]] = File(None),
                       previous: Optional[UploadFile] = File(None)):
        directory = engine.comfy / 'input' / 'h3_remote' / uuid.uuid4().hex
        directory.mkdir(parents=True, exist_ok=True)
        media = {'images': [], 'videos': [], 'audios': []}
        try:
            cfg = normalize(json.loads(settings), engine.defaults)
            limits = {'images': (9, 50*2**20), 'videos': (3, 1024*2**20), 'audios': (3, 256*2**20)}
            async def save(upload, limit):
                suffix = Path(upload.filename or '').suffix.lower()
                if not re.fullmatch(r'\.[a-z0-9]{1,8}', suffix):
                    raise ValueError('素材需要正常副檔名')
                dest = directory / (uuid.uuid4().hex + suffix)
                size = 0
                with dest.open('wb') as stream:
                    while chunk := await upload.read(2**20):
                        size += len(chunk)
                        if size > limit: raise ValueError('素材超出大小上限')
                        stream.write(chunk)
                if not size: raise ValueError('素材為空')
                return dest.relative_to(engine.comfy / 'input').as_posix()
            originals = {}
            for key, group in [('images', images), ('videos', videos), ('audios', audios)]:
                count, limit = limits[key]
                if len(group or []) > count: raise ValueError(f'{key} 最多 {count} 份')
                originals[key] = [Path(u.filename or '').name for u in group or []]
                for upload in group or []: media[key].append(await save(upload, limit))
            if previous:
                media['previous'] = await save(previous, 1024*2**20)
                originals['previous'] = Path(previous.filename or '').name
            media['source_names'] = originals
            validate_media(cfg, media, prompt)
            ident = await asyncio.to_thread(engine.submit, cfg, media, prompt)
            return {'id': ident, 'status': 'queued'}
        except (ValueError, KeyError, TypeError, subprocess.CalledProcessError) as exc:
            shutil.rmtree(directory, ignore_errors=True)
            raise HTTPException(422, str(exc)) from exc
    def get_job(ident):
        try: return engine.status(ident)
        except KeyError: raise HTTPException(404, 'Unknown job')
    @app.get('/jobs/{ident}')
    def status(ident: str):
        job = get_job(ident)
        return {k: v for k, v in job.items() if k not in ('video', 'info')}
    @app.get('/result/{ident}')
    def result(ident: str):
        job = get_job(ident)
        if job['status'] != 'completed': raise HTTPException(409, 'Job not completed')
        return FileResponse(job['video'], media_type='video/mp4', filename=Path(job['video']).name)
    @app.get('/info/{ident}')
    def info_file(ident: str):
        job = get_job(ident)
        if job['status'] != 'completed': raise HTTPException(409, 'Job not completed')
        return FileResponse(job['info'], media_type='text/plain; charset=utf-8')
    @app.get('/workflow/{ident}')
    def workflow(ident: str):
        get_job(ident)
        return FileResponse(engine.root / ident / 'workflow.json', media_type='application/json')
    return app


## Cell 3 — 模型、LoRA、採樣與串接設定
模型來源可選 HF repo/file、URL、本機／Drive 路徑或 installed。可刷新自訂 repo 清單。
HF_TOKEN 放 Colab Secrets 或按隱藏輸入；讀取 token 不會登入或寫出 credential。

勾選 LoRA 後填 JSON 陣列，依序套用。每項可用 `repo`+`file`、`url`、`local` 或 installed `name`。
`name` 可另存以避免同名來源衝突；`strength` 可調。
例如 `[ {"repo":"Comfy-Org/MiniMax-H3", "file":"loras/請選實際LoRA檔名.safetensors", "strength":1.0} ]`。
自訂 encoder / VAE 可在「Encoder / VAE 來源」填 JSON，key 為 text_encoder、video_vae、audio_vae；
各項可填 repo/file、url、local，或 `{"name":"已安裝檔名"}`。未覆寫者使用官方來源。

降低顯存只控制 KJNodes 分塊；Kitchen 與 Sol 各有獨立開關。硬體不支援 sparse kernel 時可能使用 dense fallback。
前段起點／終點均以秒表示，終點空白代表影片結尾。開關關閉時不用上傳前段。


In [ ]:
"""Cell 3: manual controls and model preparation specifications."""
import copy
import getpass
import json
import math
from urllib.parse import urlparse, unquote
import os
from pathlib import Path

import ipywidgets as W
from IPython.display import display
from huggingface_hub import HfApi


# 僅保存介面設定，不保存 HF_TOKEN / API_TOKEN。
import threading
H3_STATE_PATH = Path(globals().get("H3_STATE_PATH", "/content/h3_colab_ui_state.json"))
_H3_STATE_LOCK = globals().get("_H3_STATE_LOCK") or threading.RLock()

def _h3_state_paths():
    paths = [H3_STATE_PATH]
    drive = Path("/content/drive/MyDrive")
    if drive.is_dir():
        paths.append(drive / "H3_Colab" / "ui_state.json")
    return list(dict.fromkeys(paths))

if "_H3_UI_STATE" not in globals():
    _H3_UI_STATE = {}
    _h3_candidates = []
    for _p in _h3_state_paths():
        try:
            _data = json.loads(_p.read_text(encoding="utf-8"))
            if isinstance(_data, dict):
                _h3_candidates.append((_p.stat().st_mtime, _data))
        except (OSError, ValueError):
            pass
    if _h3_candidates:
        _H3_UI_STATE = max(_h3_candidates, key=lambda item: item[0])[1]

def _h3_save_state(section, value):
    with _H3_STATE_LOCK:
        _H3_UI_STATE[section] = copy.deepcopy(value)
        payload = json.dumps(_H3_UI_STATE, ensure_ascii=False, indent=2)
        errors = []
        for dest in _h3_state_paths():
            try:
                dest.parent.mkdir(parents=True, exist_ok=True)
                temporary = dest.with_name(dest.name + ".tmp")
                temporary.write_text(payload, encoding="utf-8")
                temporary.replace(dest)
            except OSError as exc:
                errors.append(f"{dest}: {exc}")
        if errors and not globals().get("_H3_STATE_WARNED"):
            globals()["_H3_STATE_WARNED"] = True
            print("部分設定檔無法寫入，runtime 記憶體仍保留設定：", *errors, sep="\n")

def _h3_widget_value(name, default=None):
    widget = globals().get(name)
    return getattr(widget, "value", default)

def _h3_snapshot3():
    return {
        "controls": {key: widget.value for key, widget in globals().get("CONTROLS", {}).items()},
        "source": _h3_widget_value("SOURCE", "hf"),
        "repo": _h3_widget_value("REPO", "Comfy-Org/MiniMax-H3"),
        "model_file": _h3_widget_value("MODEL_FILE", ""),
        "model_location": _h3_widget_value("MODEL_LOCATION", ""),
        "aux_specs": _h3_widget_value("AUX_SPECS", "{}"),
        "lora_enabled": _h3_widget_value("LORA_ENABLED", False),
        "lora_specs": _h3_widget_value("LORA_SPECS", "[]"),
        "lora_repo": _h3_widget_value("LORA_REPO", ""),
        "lora_file": _h3_widget_value("LORA_FILE", ""),
        "lora_strength": _h3_widget_value("LORA_STRENGTH", 1.0),
        "previous_end": _h3_widget_value("PREVIOUS_END", ""),
        "xet": _h3_widget_value("XET", False),
        "mp": _h3_widget_value("MP_SELECT"),
        "aspect": _h3_widget_value("ASPECT_SELECT", 16/9),
    }

# 重跑前先擷取畫面上的最新輸入，包含尚未按套用的內容。
if globals().get("CONTROLS") and "SOURCE" in globals():
    _h3_save_state("cell3", _h3_snapshot3())
_S3 = copy.deepcopy(_H3_UI_STATE.get("cell3", {}))
_H3_RESTORING3 = True

HF_TOKEN = os.environ.get('HF_TOKEN', '')
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or HF_TOKEN
except Exception:
    pass
OFFICIAL = 'Comfy-Org/MiniMax-H3'
PRESETS = {
    OFFICIAL: ['diffusion_models/minimax_h3_fl2va_int8_convrot.safetensors', 'diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors',
               'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors'],
    'TenStrip/10Eros-Max': ['10Eros_Max_h3_hybrid_beta5_int8.safetensors', '10Eros_Max_h3_TURBO-hybrid_beta5_int8.safetensors'],
    'brurpo/DaSiWa-MiniMax-H3-Hybrid': ['DasiwaMinimaxH3_dasiwaHybrid8turboV1.safetensors'],
    'WarmBloodAban/Minimax-h3_Singularity': [],
}
STYLE = {'description_width': '180px'}
LAYOUT = W.Layout(width='98%')
CONTROLS = {}
LABELS = {
    'mode': '生成模式', 'duration': '採樣秒數（裁切前）', 'width': '寬度', 'height': '高度', 'seed': 'Seed（空白隨機）',
    'steps': 'Steps', 'sampler': 'Sampler', 'scheduler': 'Scheduler', 'model': 'Diffusion 模型檔名',
    'text_encoder': 'Text encoder', 'video_vae': 'Video VAE', 'audio_vae': 'Audio VAE',
    'shift_enabled': '使用自訂 Video / Audio Shift', 'shift_video': 'Video shift', 'shift_audio': 'Audio shift',
    'kitchen_attention': 'Kitchen attention', 'sol_attention': 'Sol attention', 'low_vram': '降低顯存',
    'head_chunks': 'Attention head chunks', 'ff_chunks': 'Feedforward chunks', 'seq_threshold': 'FF seq threshold',
    'motion_context': '串接前段影片', 'context_length': '影像 context 影格', 'audio_context_length': '音訊 context 影格',
    'previous_start': '選取區間起點（秒）', 'prefix': '輸出前綴', 'ref_image_size': '參考影像尺寸',
}
def control(key):
    val = _S3.get('controls', {}).get(key, globals().get('SETTINGS', {}).get(key, DEFAULTS[key]))
    options = {'mode': ['ref2va', 't2va', 'i2va', 'fl2va', 'l2va'],
               'context_length': [5, 22, 39, 56], 'ref_image_size': ['match', 'max'],
               'reference_video_policy': ['official_2_to_15s', 'model_minimum']}
    args = dict(value=val, description=LABELS.get(key, key), style=STYLE, layout=W.Layout(width='98%'))
    if key in options: obj = W.Dropdown(options=options[key], **args)
    elif type(val) is bool: obj = W.Checkbox(**args)
    elif type(val) is int: obj = W.IntText(**args)
    elif type(val) is float: obj = W.FloatText(**args)
    else: obj = W.Text(**args)
    CONTROLS[key] = obj
    return obj

SOURCE = W.Dropdown(options=['hf', 'url', 'local', 'installed'], description='模型來源', style=STYLE)
REPO = W.Combobox(options=list(PRESETS), value=OFFICIAL, description='HF repo', ensure_option=False, layout=W.Layout(width="98%"), style=STYLE)
MODEL_FILE = W.Combobox(options=PRESETS[OFFICIAL], value=PRESETS[OFFICIAL][0], description='HF file', ensure_option=False, layout=W.Layout(width="98%"), style=STYLE)
MODEL_LOCATION = W.Text(description='URL 或本機路徑', layout=W.Layout(width="98%"), style=STYLE)
REFRESH = W.Button(description='刷新 repo 模型清單')
AUTH = W.Button(description='隱藏輸入 HF token')
MOUNT = W.Button(description='掛載 Google Drive')
XET = W.Checkbox(value=False, description='Xet 高效能下載')
LORA_ENABLED = W.Checkbox(value=False, description='啟用有序多 LoRA')
LORA_SPECS = W.Textarea(value='[]', description='LoRA JSON', layout=W.Layout(width='98%', height='180px'), style=STYLE)
AUX_SPECS = W.Textarea(value='{}', description='Encoder / VAE 來源',
                       placeholder='可選：{"text_encoder":{"local":"/content/drive/MyDrive/encoder.safetensors"}}',
                       layout=W.Layout(width='98%', height='100px'), style=STYLE)
ADD_LORA = W.Button(description='新增 LoRA 欄位')
PREVIOUS_END = W.Text(value='', description='選取區間終點（秒）', placeholder='空白到影片末尾', layout=W.Layout(width="98%"), style=STYLE)
OUT = W.Output()
APPLY = W.Button(description='套用全部設定', button_style='success')
def refresh(_):
    with OUT:
        OUT.clear_output()
        try:
            files = HfApi(token=HF_TOKEN or False).list_repo_files(REPO.value)
            MODEL_FILE.options = [x for x in files if x.endswith('.safetensors')]
            print('清單已更新；請選倉庫內的實際模型檔案，使用檔名會自動帶入。')
        except Exception as exc: print(type(exc).__name__, '：無法讀取 repo，請確認名稱與 HF 權限。')
def repo_changed(change):
    choices = PRESETS.get(REPO.value, [])
    MODEL_FILE.options = choices
    MODEL_FILE.value = choices[0] if choices else ''
def file_changed(change):
    if MODEL_FILE.value: CONTROLS['model'].value = Path(MODEL_FILE.value).name
def auth(_):
    global HF_TOKEN
    HF_TOKEN = getpass.getpass('HF token（只保留在 runtime 記憶體）: ').strip()
def mount(_):
    from google.colab import drive
    drive.mount('/content/drive')
def add_lora(_):
    with OUT:
        try:
            specs = json.loads(LORA_SPECS.value)
            specs.append({'repo': OFFICIAL, 'file': 'loras/填入檔名.safetensors', 'strength': 1.0})
            LORA_SPECS.value = json.dumps(specs, ensure_ascii=False, indent=2)
        except Exception: print('請先修正 LoRA JSON 陣列')
def apply_settings(_=None):
    global SETTINGS, DOWNLOADS
    with OUT:
        OUT.clear_output()
        try:
            if globals().get('ENGINE') and ENGINE.queue.unfinished_tasks:
                raise RuntimeError('請等目前工作完成後再套用設定')
            sync_model_name()
            cfg = {k: w.value for k, w in CONTROLS.items()}
            cfg['previous_end'] = float(PREVIOUS_END.value) if PREVIOUS_END.value.strip() else None
            loras = json.loads(LORA_SPECS.value) if LORA_ENABLED.value else []
            selected = []
            downloads = []
            for spec in loras:
                name = spec.get('name') or Path(spec.get('file') or spec.get('local') or spec.get('url', '').split('?')[0]).name
                name = safe_name(name)
                selected.append({'name': name, 'strength': spec.get('strength', 1.0)})
                downloads.append(('loras', name, spec))
            cfg['loras'] = selected
            normalize(cfg)  # validate without consuming the blank random seed setting
            if SOURCE.value == 'hf':
                if not REPO.value.strip() or not MODEL_FILE.value.strip(): raise ValueError('請填 HF 倉庫與檔案路徑')
                spec = {'repo': REPO.value.strip(), 'file': MODEL_FILE.value.strip()}
            elif SOURCE.value == 'url':
                if urlparse(MODEL_LOCATION.value).scheme not in ('http','https'): raise ValueError('請填 http/https 網址')
                spec = {'url': MODEL_LOCATION.value.strip()}
            elif SOURCE.value == 'local':
                if not MODEL_LOCATION.value.strip(): raise ValueError('請填完整路徑')
                spec = {'local': MODEL_LOCATION.value.strip()}
            else: spec = {'name': cfg['model']}
            downloads = [('diffusion_models', cfg['model'], spec)] + downloads
            auxiliary = json.loads(AUX_SPECS.value)
            if not isinstance(auxiliary, dict) or set(auxiliary) - {'text_encoder', 'video_vae', 'audio_vae'}:
                raise ValueError('Encoder / VAE 來源只接受 text_encoder、video_vae、audio_vae')
            for folder, key in [('text_encoders', 'text_encoder'), ('vae', 'video_vae'), ('vae', 'audio_vae')]:
                downloads.append((folder, cfg[key], auxiliary.get(key, {'repo': OFFICIAL, 'file': f'{folder}/{cfg[key]}'})))
            SETTINGS, DOWNLOADS = cfg, downloads
            _h3_save_state('cell3', _h3_snapshot3())
            os.environ['HF_XET_HIGH_PERFORMANCE'] = '1' if XET.value else '0'
            if globals().get('ENGINE'): ENGINE.defaults = copy.deepcopy(SETTINGS)
            print('設定已套用並保存。模型與 LoRA 變更後請執行 Cell 4。')
            if SOURCE.value == 'hf': print('實際來源：', REPO.value, '/', MODEL_FILE.value)
            print('儲存並使用：', cfg['model'])
            print('串接:', cfg['motion_context'], '| 降低顯存:', cfg['low_vram'], '| Kitchen:', cfg['kitchen_attention'], '| Sol:', cfg['sol_attention'])
        except Exception as exc: print(type(exc).__name__, str(exc))

groups = [
    ('模型與下載', ['mode', 'model', 'text_encoder', 'video_vae', 'audio_vae']),
    ('採樣與輸出', ['width', 'height', 'duration', 'steps', 'sampler', 'scheduler', 'seed', 'prefix', 'crf',
                   'shift_enabled', 'shift_video', 'shift_audio', 'ref_image_size', 'reference_video_policy']),
    ('Attention', ['kitchen_attention', 'sol_attention', 'sol_tau', 'sol_start', 'sol_end', 'sol_dense_blocks',
                   'sol_min_tokens', 'sol_extra_tokens', 'sol_verbose']),
    ('降低顯存', ['low_vram', 'head_chunks', 'ff_chunks', 'seq_threshold']),
    ('Motion Context', ['motion_context', 'context_length', 'audio_context_length', 'previous_start']),
]

def _text(value='', description='', **kw):
    return W.Text(value=value, description=description, style=STYLE, layout=W.Layout(width='98%'), **kw)
for _, keys in groups:
    for key in keys: control(key)

# 還原來源欄位；Observer 尚未接上，不會覆蓋保存的選擇。
SOURCE.value = _S3.get("source", "hf")
REPO.value = _S3.get("repo", OFFICIAL)
MODEL_FILE.options = PRESETS.get(REPO.value, [])
MODEL_FILE.value = _S3.get("model_file") or (MODEL_FILE.options[0] if MODEL_FILE.options else "")
MODEL_LOCATION.value = _S3.get("model_location", "")
AUX_SPECS.value = _S3.get("aux_specs", "{}")
LORA_ENABLED.value = _S3.get("lora_enabled", False)
LORA_SPECS.value = _S3.get("lora_specs", "[]")
PREVIOUS_END.value = _S3.get("previous_end", "")
XET.value = _S3.get("xet", False)

# ── 清楚區分下載來源與實際使用檔名 ──
SOURCE.options = [("Hugging Face 倉庫下載", "hf"), ("下載網址", "url"),
                  ("本機 / Google Drive 檔案", "local"), ("已安裝模型", "installed")]
REPO.description = "① HF 倉庫"
MODEL_FILE.description = "② 倉庫內檔案"
CONTROLS["model"].description = "實際使用檔名"

def sync_model_name(_=None):
    source = SOURCE.value
    REPO.layout.display = "" if source == "hf" else "none"
    MODEL_FILE.layout.display = "" if source == "hf" else "none"
    MODEL_LOCATION.layout.display = "" if source in ("url", "local") else "none"
    CONTROLS["model"].disabled = source != "installed"
    if source == "hf":
        name = Path(MODEL_FILE.value.strip()).name
    elif source == "url":
        name = Path(unquote(urlparse(MODEL_LOCATION.value.strip()).path)).name
    elif source == "local":
        name = Path(MODEL_LOCATION.value.strip()).name
    else:
        return
    CONTROLS["model"].value = name

MP_SELECT = W.Dropdown(options=[("自訂：直接填寬高", None)] + [(f"{i/10:.1f} MP", i/10) for i in range(2,11)],
                       value=_S3.get("mp"), description="解析度大小", style=STYLE, layout=W.Layout(width="98%"))
ASPECT_SELECT = W.Dropdown(options=[("16:9 橫式",16/9),("9:16 直式",9/16),("1:1 正方形",1.0),("4:3 橫式",4/3),("3:4 直式",3/4)],
                           value=_S3.get("aspect",16/9), description="畫面比例", style=STYLE, layout=W.Layout(width="98%"))
SIZE_INFO = W.HTML()
_H3_SIZE_BUSY = False
def show_size():
    w,h = CONTROLS["width"].value,CONTROLS["height"].value
    SIZE_INFO.value = f"<small>目前 {w} × {h}，約 {w*h/1e6:.3f} MP；選單會對齊 32 的倍數。</small>"
def choose_size(_=None):
    global _H3_SIZE_BUSY
    if _H3_SIZE_BUSY or MP_SELECT.value is None: return
    _H3_SIZE_BUSY = True
    try:
        pixels,ratio = MP_SELECT.value*1e6,ASPECT_SELECT.value
        CONTROLS["width"].value = max(32,round(math.sqrt(pixels*ratio)/32)*32)
        CONTROLS["height"].value = max(32,round(math.sqrt(pixels/ratio)/32)*32)
    finally: _H3_SIZE_BUSY = False
    show_size()
def manual_size(_=None):
    if not _H3_SIZE_BUSY: MP_SELECT.value = None
    show_size()

LORA_REPO = _text("", "LoRA HF 倉庫", placeholder="作者/倉庫名稱")
LORA_FILE = _text("", "LoRA 檔案路徑", placeholder="倉庫內完整相對路徑，含 .safetensors")
LORA_STRENGTH = W.FloatText(value=_S3.get("lora_strength",1.0),description="LoRA 強度",style=STYLE)
LORA_REPO.value = _S3.get("lora_repo","")
LORA_FILE.value = _S3.get("lora_file","")
def add_lora(_):
    with OUT:
        try:
            if not LORA_REPO.value.strip() or not LORA_FILE.value.strip():
                raise ValueError("請填 LoRA 倉庫與檔案路徑")
            specs = json.loads(LORA_SPECS.value)
            if not isinstance(specs,list): raise ValueError("LoRA 清單必須是 [] 陣列")
            specs.append({"repo":LORA_REPO.value.strip(),"file":LORA_FILE.value.strip(),"strength":LORA_STRENGTH.value})
            LORA_ENABLED.value=True
            LORA_SPECS.value=json.dumps(specs,ensure_ascii=False,indent=2)
            print("已加入 LoRA，最後按套用設定。")
        except Exception as exc: print(str(exc))

def _note(text): return W.HTML('<div style="line-height:1.7;padding:8px 0">'+text+'</div>')
MODEL_HELP = _note("<b>主模型：</b>① HF 倉庫是「作者/倉庫名稱」，例如 Comfy-Org/MiniMax-H3。<br>② 倉庫內檔案是實際下載路徑，包含 diffusion_models/ 等資料夾。<br>實際使用檔名自動帶入；只改名字不會改變下載模型。<br>Text encoder / VAE 通常保留預設；來源 JSON 沒有要改就維持 {}。")
LORA_HELP = _note("<b>LoRA：</b>填作者/倉庫、倉庫內完整檔案路徑與強度，按新增即可自動產生 JSON。<br>多份按照清單從上到下套用；可直接編輯 JSON 調整順序或移除項目。強度可先用 1.0，再按作者建議調整。")
SHIFT_HELP = _note("<b>Shift：</b>取消勾選「使用自訂 Video / Audio Shift」會略過 Shift 節點、沿用模型設定；不用填 0。此開關同時控制兩項自訂值。")
STATE_HELP = _note("<b>自動保留輸入：</b>重跑 Cell 3 / 6 會恢復上次設定。修改生成參數後仍需按「套用全部設定」。<br>設定保存在 /content/h3_colab_ui_state.json；掛載 Drive 後也會備份到 MyDrive/H3_Colab/ui_state.json。<br>要跨 runtime 恢復，請先掛載 Drive，再執行本格；/content 的素材與成品不會因保存設定而自動備份。")

boxes = [
    W.VBox([STATE_HELP, MODEL_HELP, CONTROLS['mode'], SOURCE, REPO, MODEL_FILE, MODEL_LOCATION, CONTROLS['model'],
            W.HBox([REFRESH,AUTH,MOUNT]),XET,CONTROLS['text_encoder'],CONTROLS['video_vae'],CONTROLS['audio_vae'],AUX_SPECS,
            LORA_HELP,LORA_ENABLED,LORA_REPO,LORA_FILE,LORA_STRENGTH,ADD_LORA,LORA_SPECS]),
    W.VBox([MP_SELECT,ASPECT_SELECT,SIZE_INFO,SHIFT_HELP]+[CONTROLS[k] for k in groups[1][1]]),
    W.VBox([CONTROLS[k] for k in groups[2][1]]),
    W.VBox([CONTROLS[k] for k in groups[3][1]]),
    W.VBox([CONTROLS[k] for k in groups[4][1]]+[PREVIOUS_END]),
]
accordion = W.Accordion(children=boxes, selected_index=0)
for i,(title,_) in enumerate(groups): accordion.set_title(i,title)

def visibility(_=None):
    for k in ('head_chunks','ff_chunks','seq_threshold'):
        CONTROLS[k].layout.display = '' if CONTROLS['low_vram'].value else 'none'
    for k in ('context_length','audio_context_length','previous_start'):
        CONTROLS[k].layout.display = '' if CONTROLS['motion_context'].value else 'none'
    PREVIOUS_END.layout.display = '' if CONTROLS['motion_context'].value else 'none'
    for k in ('shift_video','shift_audio'):
        CONTROLS[k].layout.display = '' if CONTROLS['shift_enabled'].value else 'none'
    for w in (LORA_REPO,LORA_FILE,LORA_STRENGTH,ADD_LORA,LORA_SPECS):
        w.layout.display = '' if LORA_ENABLED.value else 'none'

def save_cell3(_=None):
    if not _H3_RESTORING3: _h3_save_state('cell3',_h3_snapshot3())

REPO.observe(repo_changed,names='value')
for w in (SOURCE,MODEL_FILE,MODEL_LOCATION): w.observe(sync_model_name,names='value')
for k in ('low_vram','motion_context','shift_enabled'): CONTROLS[k].observe(visibility,names='value')
LORA_ENABLED.observe(visibility,names='value')
MP_SELECT.observe(choose_size,names='value');ASPECT_SELECT.observe(choose_size,names='value')
CONTROLS['width'].observe(manual_size,names='value');CONTROLS['height'].observe(manual_size,names='value')
for w in list(CONTROLS.values())+[SOURCE,REPO,MODEL_FILE,MODEL_LOCATION,AUX_SPECS,LORA_ENABLED,LORA_SPECS,
                                 LORA_REPO,LORA_FILE,LORA_STRENGTH,PREVIOUS_END,XET,MP_SELECT,ASPECT_SELECT]:
    w.observe(save_cell3,names='value')
REFRESH.on_click(refresh);AUTH.on_click(auth);MOUNT.on_click(mount);ADD_LORA.on_click(add_lora);APPLY.on_click(apply_settings)
sync_model_name();show_size();visibility()
_H3_RESTORING3 = False
save_cell3()
display(accordion,APPLY,OUT)
apply_settings()


## Cell 4 — 準備所選模型
僅下載指定檔案。若同名權重來源不同，請另命名。

In [ ]:
"""Cell 4: download only selected model files, preserving their source identity."""
import json
import os
import shutil
import subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download

def install_artifact(folder, name, spec):
    name = safe_name(name)
    target = COMFY/'models'/folder/name
    target.parent.mkdir(parents=True, exist_ok=True)
    identity = {k: spec[k] for k in ('repo', 'file', 'revision', 'local', 'url') if spec.get(k)}
    receipt = target.with_name(target.name + '.source.json')
    if target.is_file() and target.stat().st_size:
        if not identity or receipt.is_file() and json.loads(receipt.read_text()) == identity:
            print('重用:', name)
            return
        raise ValueError(f'{name} 已存在但來源不同或未知，請另選檔名')
    if spec.get('repo') and spec.get('file'):
        source = Path(hf_hub_download(spec['repo'], spec['file'], revision=spec.get('revision', 'main'), token=HF_TOKEN or False))
    elif spec.get('local'):
        source = Path(spec['local'])
    elif spec.get('url'):
        from urllib.parse import urlparse
        if urlparse(spec['url']).scheme not in ('http', 'https'):
            raise ValueError('模型 URL 只接受 http / https')
        partial = target.with_name(target.name+'.partial')
        subprocess.run(['curl', '-L', '--fail', '--retry', '3', '-C', '-', '-o', str(partial), spec['url']], check=True)
        if not partial.stat().st_size: raise ValueError('下載為空')
        partial.replace(target)
        receipt.write_text(json.dumps(identity), encoding='utf-8')
        print('已下載:', name)
        return
    else:
        raise FileNotFoundError(f'尚未安裝: {name}')
    if not source.is_file() or not source.stat().st_size:
        raise FileNotFoundError('模型來源不存在或為空')
    if target.is_symlink(): target.unlink()
    # Hard links are visible to ComfyUI's model discovery and avoid duplicating HF cache data.
    try: os.link(source.resolve(), target)
    except OSError: shutil.copyfile(source, target)
    receipt.write_text(json.dumps(identity), encoding='utf-8')
    print('已準備:', name)

assert 'DOWNLOADS' in globals(), '先套用 Cell 3 設定'
if globals().get('ENGINE') and ENGINE.queue.unfinished_tasks:
    raise RuntimeError('請先等工作完成')
for folder, name, spec in DOWNLOADS:
    install_artifact(folder, name, spec)
print('模型準備完成，執行 Cell 5。')


## Cell 5 — 啟動 ComfyUI
ComfyUI 綁定 localhost。安裝節點後必須重啟服務。生成中禁止重啟。

In [ ]:
"""Cell 5: start private ComfyUI and shared generation engine."""
import importlib.metadata
import subprocess
import sys
import time
import requests

if globals().get('ENGINE') and ENGINE.queue.unfinished_tasks:
    raise RuntimeError('有未完成工作，請勿重啟服務')
if globals().get('API_SERVER'):
    API_SERVER.should_exit = True
    API_THREAD.join(timeout=10)
    if API_THREAD.is_alive(): raise RuntimeError('API 尚未停止')
if globals().get('ENGINE'):
    ENGINE.queue.put(None)
    ENGINE.thread.join(timeout=5)
if globals().get('COMFY_PROCESS') and COMFY_PROCESS.poll() is None:
    COMFY_PROCESS.terminate()
    COMFY_PROCESS.wait(timeout=30)
LOG_PATH = Path('/content/comfyui_h3.log')
LOG_STREAM = LOG_PATH.open('a')
# low_vram is the KJNodes group, independent from ComfyUI's model offloading policy.
COMFY_PROCESS = subprocess.Popen([sys.executable, str(COMFY/'main.py'), '--listen', '127.0.0.1',
                                  '--port', '8188', '--disable-auto-launch'], cwd=COMFY,
                                 stdout=LOG_STREAM, stderr=subprocess.STDOUT)
for attempt in range(180):
    if COMFY_PROCESS.poll() is not None:
        raise RuntimeError(LOG_PATH.read_text(errors='replace')[-8000:])
    try:
        response = requests.get('http://127.0.0.1:8188/system_stats', timeout=2)
        response.raise_for_status()
        break
    except requests.RequestException: time.sleep(2)
else: raise TimeoutError('ComfyUI 啟動逾時，查看 /content/comfyui_h3.log')
PROVENANCE['comfy_kitchen'] = importlib.metadata.version('comfy-kitchen')
try:
    import comfy_kitchen
    available = bool(comfy_kitchen.sol_attn_is_available(torch.device('cuda')))
except Exception:
    available = False
PROVENANCE['sol_kernel_available'] = available
print('Sol kernel available:', available)
if not available:
    print('Sol 節點仍依設定啟用；此 GPU/runtime 可能使用 dense fallback，查看 ComfyUI log。')
ENGINE = Engine(COMFY, defaults=SETTINGS, provenance=PROVENANCE)
info = ENGINE.info()
required = {'MiniMaxH3AudioConditioningT8', 'MiniMaxH3MotionContext', 'MiniMaxH3MotionContextTrim',
            'MiniMaxLowVRAMAttention', 'MiniMaxChunkFeedForward', 'ModelAttentionBackend', 'BlockSparseAttention',
            'VHS_LoadVideo', 'VHS_VideoCombine'}
if required - set(info):
    raise RuntimeError(f'節點載入失敗: {sorted(required-set(info))}；查看 {LOG_PATH}')
print('ComfyUI 與生成引擎已就緒。手動操作跑 Cell 6；遠端 API 跑 Cell 7。')


## Cell 6 — 手動上傳、預覽、生成與下載
Pictures / 參考 Videos / Audios 各自依路徑欄逐行排序。可直接填 Colab 或 Drive 路徑。
前段用獨立上傳按鈕，也能選擇任一已完成片段。選擇後請檢查 Cell 3 的模式／區間並套用。
開始下一段時先修改 prompt；不會自動沿用上一段作前段。
停止等待不代表 GPU 停止；請勿重送同一工作，可用 `ENGINE.status(LAST_MANUAL_JOB)` 查詢。
下載按鈕保存 MP4 與伺服器生成 TXT；可另存 Drive。


In [ ]:
"""Cell 6: upload, preview, export, generate, download and continue any chosen clip."""
import copy
import sys
import subprocess
import datetime
import json
import shutil
import time
import uuid
from pathlib import Path
import ipywidgets as W
from IPython.display import display, Video, Audio, Image


# 僅保存介面設定，不保存 HF_TOKEN / API_TOKEN。
import threading
H3_STATE_PATH = Path(globals().get("H3_STATE_PATH", "/content/h3_colab_ui_state.json"))
_H3_STATE_LOCK = globals().get("_H3_STATE_LOCK") or threading.RLock()

def _h3_state_paths():
    paths = [H3_STATE_PATH]
    drive = Path("/content/drive/MyDrive")
    if drive.is_dir():
        paths.append(drive / "H3_Colab" / "ui_state.json")
    return list(dict.fromkeys(paths))

if "_H3_UI_STATE" not in globals():
    _H3_UI_STATE = {}
    _h3_candidates = []
    for _p in _h3_state_paths():
        try:
            _data = json.loads(_p.read_text(encoding="utf-8"))
            if isinstance(_data, dict):
                _h3_candidates.append((_p.stat().st_mtime, _data))
        except (OSError, ValueError):
            pass
    if _h3_candidates:
        _H3_UI_STATE = max(_h3_candidates, key=lambda item: item[0])[1]

def _h3_save_state(section, value):
    with _H3_STATE_LOCK:
        _H3_UI_STATE[section] = copy.deepcopy(value)
        payload = json.dumps(_H3_UI_STATE, ensure_ascii=False, indent=2)
        errors = []
        for dest in _h3_state_paths():
            try:
                dest.parent.mkdir(parents=True, exist_ok=True)
                temporary = dest.with_name(dest.name + ".tmp")
                temporary.write_text(payload, encoding="utf-8")
                temporary.replace(dest)
            except OSError as exc:
                errors.append(f"{dest}: {exc}")
        if errors and not globals().get("_H3_STATE_WARNED"):
            globals()["_H3_STATE_WARNED"] = True
            print("部分設定檔無法寫入，runtime 記憶體仍保留設定：", *errors, sep="\n")

def _h3_widget_value(name, default=None):
    widget = globals().get(name)
    return getattr(widget, "value", default)

try:
    import websocket
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','websocket-client'])
    import websocket

def _h3_snapshot6():
    old = _H3_UI_STATE.get('cell6',{})
    selected = _h3_widget_value('JOBS_SELECT',old.get('selected_video'))
    if selected and str(selected) in getattr(globals().get('ENGINE'),'jobs',{}):
        selected = ENGINE.status(selected).get('video')
    return {
        'prompt': _h3_widget_value('PROMPT',''),
        'paths': {k:w.value for k,w in globals().get('PATHS',{}).items()},
        'previous': _h3_widget_value('PREVIOUS_PATH',''),
        'drive_output': _h3_widget_value('DRIVE_OUTPUT',''),
        'selected_video': selected,
        'last_job': globals().get('LAST_MANUAL_JOB',old.get('last_job')),
        'last_video': globals().get('LAST_MANUAL_VIDEO',old.get('last_video')),
        'generated_files': list(old.get('generated_files',[])),
    }
if 'PROMPT' in globals() and globals().get('PATHS'):
    _h3_save_state('cell6',_h3_snapshot6())
_S6 = copy.deepcopy(_H3_UI_STATE.get('cell6',{}))
_H3_RESTORING6 = True
import json
import math
import threading
import time
from collections import deque
from urllib.parse import urlencode, urlsplit, urlunsplit


def _step_time_text(seconds):
    if seconds is None:
        return '估算中'
    seconds = max(0, int(round(seconds)))
    hours, rest = divmod(seconds, 3600)
    minutes, seconds = divmod(rest, 60)
    if hours:
        return f'{hours} 小時 {minutes:02d} 分 {seconds:02d} 秒'
    if minutes:
        return f'{minutes} 分 {seconds:02d} 秒'
    return f'{seconds} 秒'


class _ComfyStepProgress:
    def __init__(self, job_id, base_url, graph=None, clock=time.monotonic):
        self.job_id = str(job_id)
        self.base_url = base_url.rstrip('/')
        self.graph = graph or {}
        self.clock = clock
        self.prompt_id = None
        self.node = None
        self.value = None
        self.maximum = None
        self.samples = deque(maxlen=10)
        self.connected = False
        self.error = False
        self.lock = threading.RLock()
        self.stop_event = threading.Event()
        self.thread = None
        self.ws = None

    def ws_url(self):
        parts = urlsplit(self.base_url)
        # Match the client_id used by _remote_execute_graph in Cell 6.
        # A separate random ID would not receive this prompt's step events.
        return urlunsplit((
            'wss' if parts.scheme == 'https' else 'ws',
            parts.netloc,
            parts.path.rstrip('/') + '/ws',
            urlencode({'clientId': self.job_id}),
            '',
        ))

    def set_prompt(self, prompt_id):
        if prompt_id:
            with self.lock:
                self.prompt_id = str(prompt_id)

    def _change_node(self, node):
        node = str(node) if node is not None else None
        if node != self.node:
            self.node = node
            self.value = self.maximum = None
            self.samples.clear()

    def _update(self, node, value, maximum):
        try:
            value, maximum = float(value), float(maximum)
        except (ValueError, TypeError):
            return
        if not math.isfinite(value) or not math.isfinite(maximum) or maximum <= 0 or value < 0:
            return
        self._change_node(node)
        value = min(value, maximum)
        if self.maximum != maximum or self.value is not None and value < self.value:
            self.samples.clear()
        self.maximum = maximum
        if self.value != value:
            self.samples.append((self.clock(), value))
        self.value = value

    def consume(self, event):
        """Support both current progress_state and older progress messages."""
        if not isinstance(event, dict) or not isinstance(event.get('data'), dict):
            return
        data = event['data']
        prompt = data.get('prompt_id')
        with self.lock:
            if prompt and (not self.prompt_id or str(prompt) != self.prompt_id):
                return
            kind = event.get('type')
            if kind == 'executing':
                self._change_node(data.get('node'))
            elif kind == 'progress':
                node = data.get('node', self.node)
                if node is not None:
                    self._update(node, data.get('value'), data.get('max'))
            elif kind == 'progress_state':
                nodes = data.get('nodes', {})
                if not isinstance(nodes, dict):
                    return
                running = [(key, state) for key, state in nodes.items()
                           if isinstance(state, dict) and state.get('state') == 'running']
                if running:
                    key, state = running[-1]
                    state_prompt = state.get('prompt_id')
                    if state_prompt and str(state_prompt) != self.prompt_id:
                        return
                    node = state.get('display_node_id') or state.get('real_node_id') or state.get('node_id') or key
                    self._update(node, state.get('value'), state.get('max'))
                elif self.node is not None:
                    for key, state in nodes.items():
                        if not isinstance(state, dict):
                            continue
                        node = state.get('display_node_id') or state.get('real_node_id') or state.get('node_id') or key
                        if str(node) == self.node and state.get('state') == 'finished':
                            self._update(node, state.get('max'), state.get('max'))

    def snapshot(self):
        with self.lock:
            node, value, maximum = self.node, self.value, self.maximum
            samples = list(self.samples)
            connected, error = self.connected, self.error
        specification = self.graph.get(node, {})
        kind = specification.get('class_type', '')
        samplers = [key for key, entry in self.graph.items()
                    if entry.get('class_type') in ('SamplerCustomAdvanced', 'SamplerCustom', 'KSampler', 'KSamplerAdvanced')]
        unit = '進度'
        if node in samplers:
            ordinal = samplers.index(node) + 1
            label = 'H3 Sampling' if len(samplers) == 1 else f'H3 第 {ordinal} 採'
            unit = 'step'
        else:
            label = kind or (f'節點 {node}' if node else '等待執行節點')
        seconds_per_unit = eta = None
        if value is not None and maximum is not None:
            if value >= maximum:
                eta = 0.0
            elif len(samples) >= 2:
                elapsed = samples[-1][0] - samples[0][0]
                advanced = samples[-1][1] - samples[0][1]
                if elapsed > 0 and advanced > 0:
                    seconds_per_unit = elapsed / advanced
                    eta = (maximum - value) * seconds_per_unit
        return dict(node=node, label=label, unit=unit, value=value, maximum=maximum,
                    seconds_per_unit=seconds_per_unit, eta=eta, connected=connected, error=error)

    def _worker(self, websocket):
        while not self.stop_event.is_set():
            ws = None
            try:
                ws = websocket.create_connection(self.ws_url(), timeout=5,
                                                  http_proxy_host=None, http_proxy_port=None)
                ws.settimeout(1)
                with self.lock:
                    self.ws, self.connected, self.error = ws, True, False
                while not self.stop_event.is_set():
                    try:
                        message = ws.recv()
                    except websocket.WebSocketTimeoutException:
                        continue
                    if not message:
                        raise ConnectionError('WebSocket closed')
                    if isinstance(message, bytes):
                        continue  # Binary image previews are unrelated to ETA.
                    try:
                        event = json.loads(message)
                    except (ValueError, TypeError):
                        continue
                    self.consume(event)
            except Exception:
                with self.lock:
                    self.error = not self.stop_event.is_set()
            finally:
                with self.lock:
                    self.connected, self.ws = False, None
                if ws is not None:
                    try:
                        ws.close()
                    except Exception:
                        pass
            self.stop_event.wait(2)

    def start(self, websocket):
        self.thread = threading.Thread(target=self._worker, args=(websocket,), daemon=True)
        self.thread.start()

    def close(self):
        self.stop_event.set()
        with self.lock:
            ws = self.ws
        if ws is not None:
            try:
                ws.close()
            except Exception:
                pass
        if self.thread is not None:
            self.thread.join(timeout=2)

MANUAL_MEDIA = {'images': [], 'videos': [], 'audios': []}
PROMPT = W.Textarea(value=_S6.get('prompt',''), description='H3 prompt', layout=W.Layout(width='98%', height='260px'))
PATHS = {key: W.Textarea(value=_S6.get('paths',{}).get(key,''), description=label, layout=W.Layout(width='98%', height='75px'))
         for key, label in [('images', 'Pictures 順序'), ('videos', 'Videos 順序'), ('audios', 'Audios 順序')]}
PREVIOUS_PATH = W.Text(value=_S6.get('previous',''), description='前段影片路徑', layout=W.Layout(width='98%'))
MANUAL_OUT = W.Output()
JOBS_SELECT = W.Dropdown(options=[], description='已完成片段', layout=W.Layout(width='98%'))
USE_JOB = W.Button(description='選此片段作前段')
RUN = W.Button(description='生成並等待', button_style='success')
RESUME = W.Button(description='繼續監看上次工作')
REFRESH_VIDEOS = W.Button(description='重新搜尋生成影片')
PROGRESS_BAR = W.FloatProgress(value=0,min=0,max=100,description='採樣進度',layout=W.Layout(width='98%'))
PROGRESS_TEXT = W.HTML(value='尚未開始；採樣取得兩個進度點後估算剩餘時間。')
EXPORT = W.Button(description='只匯出 API workflow')
PREVIEW = W.Button(description='預覽與標籤對照')
DOWNLOAD = W.Button(description='下載最近 MP4 + TXT')
DRIVE_OUTPUT = W.Text(value=_S6.get('drive_output',''), description='另存 Drive 目錄', placeholder='空白不複製；例如 /content/drive/MyDrive/H3', layout=W.Layout(width='98%'))

def import_media(path):
    source = Path(path).expanduser().resolve()
    if not source.is_file(): raise FileNotFoundError(str(source))
    inputs = (COMFY/'input').resolve()
    if source.is_relative_to(inputs): return source.relative_to(inputs).as_posix()
    dest = inputs/'h3_manual'/f'{uuid.uuid4().hex}{source.suffix.lower()}'
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, dest)
    return dest.relative_to(inputs).as_posix()
def selected_media():
    media = {k: [import_media(p.strip()) for p in box.value.splitlines() if p.strip()] for k, box in PATHS.items()}
    media['source_names'] = {k: [Path(p.strip()).name for p in box.value.splitlines() if p.strip()] for k, box in PATHS.items()}
    # The checkbox is authoritative: stale previous-path text is ignored when off.
    if SETTINGS['motion_context']:
        if not PREVIOUS_PATH.value.strip(): raise ValueError('請上傳或選擇前段影片')
        media['previous'] = import_media(PREVIOUS_PATH.value.strip())
        media['source_names']['previous'] = Path(PREVIOUS_PATH.value).name
    validate_media(normalize(SETTINGS), media, PROMPT.value)
    ENGINE.enrich(media)
    return media
def upload(key):
    def action(_):
        from google.colab import files
        with MANUAL_OUT:
            try:
                data = files.upload()
                directory = COMFY/'input'/'h3_manual'/uuid.uuid4().hex
                directory.mkdir(parents=True, exist_ok=True)
                paths = []
                for name, payload in data.items():
                    path = directory/Path(name).name
                    path.write_bytes(payload)
                    paths.append(str(path))
                if key == 'previous':
                    if len(paths) != 1: raise ValueError('請一次選一段前段影片')
                    PREVIOUS_PATH.value = paths[0]
                    CONTROLS['motion_context'].value = True
                    apply_settings()
                else:
                    current = PATHS[key].value.strip()
                    PATHS[key].value = '\n'.join(([current] if current else []) + paths)
                print('素材已加入。請依使用者指定順序調整路徑欄，每行一個。')
            except Exception as exc: print(type(exc).__name__, str(exc))
    return action
buttons = []
for key, label in [('images', '上傳 Pictures'), ('videos', '上傳參考 Videos'), ('audios', '上傳 Audios'), ('previous', '上傳前段影片')]:
    button = W.Button(description=label)
    button.on_click(upload(key)); buttons.append(button)
def preview(_):
    with MANUAL_OUT:
        MANUAL_OUT.clear_output()
        try:
            for key, tag in [('images', 'Picture'), ('videos', 'Video'), ('audios', 'Audio')]:
                for i, path in enumerate(PATHS[key].value.splitlines(), 1):
                    if not path.strip(): continue
                    print(f'<{tag} {i}>', Path(path).name)
                    if key == 'images': display(Image(filename=path.strip(), width=360))
                    elif key == 'videos': display(Video(path.strip(), embed=True, width=480))
                    else: display(Audio(filename=path.strip()))
            if SETTINGS['motion_context'] and PREVIOUS_PATH.value.strip():
                print('Motion Context 前段（不編入 Video / Audio 參考標籤）:', PREVIOUS_PATH.value)
                display(Video(PREVIOUS_PATH.value, embed=True, width=480))
        except Exception as exc: print(type(exc).__name__, str(exc))
def export(_):
    from google.colab import files
    with MANUAL_OUT:
        try:
            cfg = normalize(SETTINGS)
            cfg['output_prefix'] = 'h3_manual/export_seed' + cfg['seed']
            media = selected_media()
            directory = COMFY/'input'/'h3_manual'/uuid.uuid4().hex
            directory.mkdir(parents=True)
            ENGINE.prepare_previous(cfg, media, directory)
            graph, _, _ = build_graph(cfg, media, PROMPT.value, ENGINE.info())
            dest = Path('/content/H3_MotionContext_api_workflow.json')
            dest.write_text(json.dumps(graph, ensure_ascii=False, indent=2), encoding='utf-8')
            print('匯出圖含此 runtime 素材路徑，換 runtime 需重新上傳。')
            files.download(str(dest))
        except Exception as exc: print(type(exc).__name__, str(exc))

def save_cell6(_=None):
    if not _H3_RESTORING6: _h3_save_state('cell6',_h3_snapshot6())

def remember_video(path):
    path = str(Path(path).resolve())
    state = copy.deepcopy(_H3_UI_STATE.get('cell6',{}))
    state['generated_files'] = list(dict.fromkeys(state.get('generated_files',[])+[path]))
    _h3_save_state('cell6',state)

def refresh_jobs(_=None):
    global _H3_RESTORING6
    state = _H3_UI_STATE.get('cell6',{})
    selected = JOBS_SELECT.value or state.get('selected_video')
    candidates = set()
    roots = [COMFY/'output']
    if DRIVE_OUTPUT.value.strip(): roots.append(Path(DRIVE_OUTPUT.value.strip()).expanduser())
    # 也搜尋設定檔預設備份目錄中的生成成品（不只依賴記憶體 jobs）。
    drive = Path('/content/drive/MyDrive/H3_Colab')
    if drive.is_dir(): roots.append(drive)
    for root in roots:
        if root.is_dir():
            for p in root.rglob('*'):
                try:
                    if p.is_file() and p.suffix.lower()=='.mp4' and p.stat().st_size:
                        candidates.add(p.resolve())
                except OSError: pass
    with ENGINE.lock:
        known = [j.get('video') for j in ENGINE.jobs.values() if j.get('status')=='completed']
    known += state.get('generated_files',[])+[state.get('last_video')]
    for filename in known:
        if filename:
            p=Path(filename).expanduser()
            if p.is_file() and p.stat().st_size: candidates.add(p.resolve())
    def modified(p):
        try:return p.stat().st_mtime
        except OSError:return 0
    ordered = sorted(candidates,key=modified,reverse=True)
    options = [('請選擇要接續的生成影片',None)]
    for p in ordered:
        stamp=datetime.datetime.fromtimestamp(modified(p)).strftime('%m/%d %H:%M')
        options.append((f'{stamp} | {p.name} | {p.parent.name}',str(p)))
    restoring = _H3_RESTORING6
    _H3_RESTORING6=True
    try:
        JOBS_SELECT.options=options
        JOBS_SELECT.value=selected if selected in [v for _,v in options] else None
    finally:_H3_RESTORING6=restoring
    save_cell6()
    return len(ordered)

def use_job(_):
    with MANUAL_OUT:
        if not JOBS_SELECT.value: print('目前沒有完成的影片'); return
        path=Path(JOBS_SELECT.value)
        if not path.is_file():
            print('此影片已不存在，請重新搜尋或指定正確路徑'); return
        PREVIOUS_PATH.value = str(path)
        CONTROLS['motion_context'].value = True
        apply_settings()
        print('已選取前段。請在 Cell 3 檢查區間與模式，再套用設定。')

# 下載事件只保留一份；保護快速連點／重複送達的 click。
import threading
import time
_H3_DOWNLOAD_LOCK = globals().get('_H3_DOWNLOAD_LOCK') or threading.Lock()
_H3_DOWNLOAD_STATE = globals().get('_H3_DOWNLOAD_STATE', {'key': None, 'at': -100.0})

def download(_):
    from google.colab import files
    if not _H3_DOWNLOAD_LOCK.acquire(blocking=False):
        return
    DOWNLOAD.disabled = True
    try:
        path = globals().get('LAST_MANUAL_VIDEO') or _H3_UI_STATE.get('cell6', {}).get('last_video')
        if not path or not Path(path).is_file():
            MANUAL_OUT.append_stdout('最近成品不存在，請確認影片路徑或掛載 Drive。\n')
            return
        video = Path(path).resolve()
        txt = video.with_suffix('.txt')
        key = (str(video), video.stat().st_mtime_ns)
        now = time.monotonic()
        if _H3_DOWNLOAD_STATE['key'] == key and now - _H3_DOWNLOAD_STATE['at'] < 3:
            return
        _H3_DOWNLOAD_STATE.update(key=key, at=now)
        # files.download 會顯示 Javascript；不要放在 with MANUAL_OUT 中。
        files.download(str(video))
        if txt.is_file():
            files.download(str(txt))
            MANUAL_OUT.append_stdout('已送出下載：1 份 MP4、1 份 TXT。\n')
        else:
            MANUAL_OUT.append_stdout('已送出 1 份 MP4；找不到同名 TXT。\n')
    except Exception as exc:
        MANUAL_OUT.append_stdout(f'下載觸發失敗：{type(exc).__name__}: {exc}\n')
    finally:
        DOWNLOAD.disabled = False
        _H3_DOWNLOAD_LOCK.release()

# 重複執行這段也只會留下一個下載 callback。
for _callback in list(DOWNLOAD._click_handlers.callbacks):
    DOWNLOAD.on_click(_callback, remove=True)
DOWNLOAD.on_click(download)

def watch_job(ident):
    global LAST_MANUAL_JOB,LAST_MANUAL_VIDEO
    tracker=None
    started=time.monotonic()
    PROGRESS_BAR.value=0;PROGRESS_BAR.bar_style=''
    RUN.disabled=RESUME.disabled=True
    try:
        job=ENGINE.status(ident)
        LAST_MANUAL_JOB=ident;save_cell6()
        path=ENGINE.root/ident/'workflow.json'
        graph=json.loads(path.read_text(encoding='utf-8')) if path.is_file() else {}
        tracker=_ComfyStepProgress(ident,ENGINE.url,graph)
        tracker.set_prompt(job.get('prompt_id'))
        tracker.start(websocket)
        print('Job:',ident)
        while True:
            job=ENGINE.status(ident)
            tracker.set_prompt(job.get('prompt_id'))
            if job['status']=='failed':raise RuntimeError(job.get('error','生成失敗'))
            if job['status']=='submission_unknown':raise RuntimeError(job.get('error','請查看 ComfyUI log，勿重送'))
            if job['status']=='completed':break
            progress=tracker.snapshot()
            elapsed=time_text(time.time()-job['submitted']) if job.get('submitted') else time_text(time.monotonic()-started)
            if job['status']!='running':
                phase={'queued':'排隊中','preparing':'準備素材與工作流'}.get(job['status'],job['status'])
                message=f'<b>{phase}</b> | 總經過時間：{elapsed}'
            elif progress['unit']=='step' and progress['maximum'] is not None:
                value,maximum=progress['value'],progress['maximum']
                PROGRESS_BAR.value=100*value/maximum
                speed=progress['seconds_per_unit']
                detail='等待更多 step，估算中' if speed is None else f"最近平均 {speed:.1f} 秒/step | 採樣約剩 {time_text(progress['eta'])}"
                if value>=maximum:detail='採樣步驟完成，等待解碼'
                message=f'<b>採樣 {value:g}/{maximum:g}</b> | 總經過時間：{elapsed}<br>{detail}'
            elif PROGRESS_BAR.value>=100:
                message=f'<b>採樣完成，解碼／寫檔中</b> | 總經過時間：{elapsed}'
            else:
                message=f"<b>{progress['label']}</b> | 總經過時間：{elapsed}<br>載入模型或等待首個採樣進度；暫不估算。"
            if progress['error']:message+='<br><small>進度連線重連中；生成工作仍由引擎執行。</small>'
            PROGRESS_TEXT.value=message
            time.sleep(0.5)
        LAST_MANUAL_VIDEO=job['video']
        remember_video(job['video']);save_cell6()
        PROGRESS_BAR.value=100;PROGRESS_BAR.bar_style='success'
        PROGRESS_TEXT.value='<b>生成完成</b>'
        print('MP4:',job['video']);print('TXT:',job['info']);print('實際輸出:',job['output'])
        display(Video(job['video'],embed=True,width=640))
        if DRIVE_OUTPUT.value.strip():
            target=Path(DRIVE_OUTPUT.value.strip()).expanduser()
            target.mkdir(parents=True,exist_ok=True)
            for key in ('video','info'):
                source=Path(job[key]);dest=target/source.name
                if dest.resolve()==source.resolve():continue
                if dest.exists():raise FileExistsError(f'拒絕覆蓋 {dest}')
                shutil.copyfile(source,dest)
                if key=='video':remember_video(dest)
            print('已另存到:',target)
        refresh_jobs()
    except KeyboardInterrupt:
        PROGRESS_TEXT.value='已停止等待，遠端 GPU 可能仍在執行；可按「繼續監看上次工作」。'
    except Exception as exc:
        PROGRESS_BAR.bar_style='danger'
        print(type(exc).__name__,str(exc))
    finally:
        if tracker:tracker.close()
        RUN.disabled=RESUME.disabled=False
        save_cell6()

def run(_):
    global LAST_MANUAL_JOB
    with MANUAL_OUT:
        MANUAL_OUT.clear_output()
        RUN.disabled=True
        try:
            existing=globals().get('LAST_MANUAL_JOB')
            if existing in ENGINE.jobs and ENGINE.status(existing)['status'] not in ('completed','failed'):
                print('上次工作仍在進行，請按「繼續監看上次工作」，不重複提交。');return
            save_cell6()
            LAST_MANUAL_JOB=ENGINE.submit(SETTINGS,selected_media(),PROMPT.value)
            save_cell6()
            watch_job(LAST_MANUAL_JOB)
        except Exception as exc:print(type(exc).__name__,str(exc))
        finally:RUN.disabled=False

def resume(_):
    with MANUAL_OUT:
        ident=globals().get('LAST_MANUAL_JOB') or _H3_UI_STATE.get('cell6',{}).get('last_job')
        if ident not in ENGINE.jobs:
            print('此引擎沒有該工作狀態，請使用生成影片選單搜尋磁碟上的成品。');return
        watch_job(ident)

def search_videos(_):
    with MANUAL_OUT:print(f'找到 {refresh_jobs()} 個 MP4。只有實際存在的檔案才會列入。')

time_text=_step_time_text
LAST_MANUAL_JOB=_S6.get('last_job')
LAST_MANUAL_VIDEO=_S6.get('last_video')
PREVIEW.on_click(preview);EXPORT.on_click(export);RUN.on_click(run)
USE_JOB.on_click(use_job);DOWNLOAD.on_click(download);RESUME.on_click(resume);REFRESH_VIDEOS.on_click(search_videos)
for w in [PROMPT,PREVIOUS_PATH,DRIVE_OUTPUT,JOBS_SELECT]+list(PATHS.values()):w.observe(save_cell6,names='value')
count=refresh_jobs()
_H3_RESTORING6=False
save_cell6()
print(f'已恢復上次輸入，找到 {count} 個生成 MP4。')
missing=[p for box in PATHS.values() for p in box.value.splitlines() if p.strip() and not Path(p.strip()).is_file()]
if PREVIOUS_PATH.value.strip() and not Path(PREVIOUS_PATH.value.strip()).is_file():missing.append(PREVIOUS_PATH.value)
if missing:print('以下保存的素材路徑目前不存在，需要重新上傳或掛載 Drive：',*missing,sep='\n')
display(W.HBox(buttons),*PATHS.values(),PREVIOUS_PATH,W.HBox([JOBS_SELECT,USE_JOB,REFRESH_VIDEOS]),PROMPT,
        DRIVE_OUTPUT,W.HBox([PREVIEW,EXPORT,RUN,RESUME,DOWNLOAD]),PROGRESS_BAR,PROGRESS_TEXT,MANUAL_OUT)


## Cell 7 — 啟動遠端 API 與 HTTPS tunnel
請在 Colab Secrets 設 H3_API_TOKEN，或執行時隱藏輸入至少 24 字元的 token；本機使用同一組。
本格顯示 H3_API_URL。API token 不會顯示，分享 notebook 前仍請清除輸出。
本機安裝交付資料夾中的 skill，設定環境變數後執行 health。每次重開 tunnel 更新 URL。
API 提供 health、models、generate、jobs、result、info、workflow；所有端點要求 Bearer token。


In [ ]:
"""Cell 7 — 驗證本機 API，建立 HTTPS tunnel，保留具體診斷。"""
import getpass
import json
import os
import re
import subprocess
import threading
import time
from pathlib import Path

import requests
import uvicorn

assert 'ENGINE' in globals() and 'create_app' in globals(), '請先完成 Cell 2–5。'
if ENGINE.queue.unfinished_tasks:
    raise RuntimeError('目前仍有生成工作，請等完成後再重啟 API / tunnel。')

API_HOST = '127.0.0.1'
API_PORT = 7860
LOCAL_API_URL = f'http://{API_HOST}:{API_PORT}'
TUNNEL_LOG = Path('/content/h3_api_tunnel.log')
# HTTP/2 使用 TCP；這是可調選項，不代表已確認你的網路阻擋 QUIC。
TUNNEL_PROTOCOL = 'http2'  # 可改成 'auto' 或 'quic'
PUBLIC_CHECK_SECONDS = 120

# 保留同一 runtime 已輸入的 token，不因重跑本格而更換。
_token = globals().get('API_TOKEN') or os.environ.get('H3_API_TOKEN', '')
if not _token:
    try:
        from google.colab import userdata
        _token = userdata.get('H3_API_TOKEN') or ''
    except Exception:
        pass
if not _token:
    _token = getpass.getpass('H3_API_TOKEN（至少 24 字元，本機使用同一組）：').strip()
if not isinstance(_token, str) or len(_token) < 24:
    raise ValueError('H3_API_TOKEN 必須至少 24 字元。')
API_TOKEN = _token
del _token


def _h3_redact(value):
    return str(value).replace(API_TOKEN, '[TOKEN REDACTED]')


def _h3_stop_tunnel(process):
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)


def _h3_log_tail():
    try:
        return _h3_redact(TUNNEL_LOG.read_text(encoding='utf-8', errors='replace')[-6000:])
    except OSError:
        return '(尚無 tunnel log)'


def _h3_check_response(response):
    if response.status_code != 200:
        body = ' '.join(response.text[:300].split())
        return False, _h3_redact(f'HTTP {response.status_code}: {body}')
    try:
        data = response.json()
    except ValueError:
        return False, 'HTTP 200，但內容不是 API JSON（可能是代理或 Cloudflare 頁面）'
    if not isinstance(data, dict) or data.get('protocol') != PROTOCOL:
        return False, 'HTTP 200，但 API protocol 不符'
    if data.get('ready') is not True:
        return False, 'API 尚未就緒'
    return True, 'HTTP 200，認證與 protocol 正確'


def _h3_public_check(session, url, timeout_seconds=PUBLIC_CHECK_SECONDS):
    deadline = time.monotonic() + timeout_seconds
    last = '尚未取得回應'
    last_report = 0.0
    while time.monotonic() < deadline:
        if TUNNEL_PROCESS.poll() is not None:
            return False, f'cloudflared 已退出，code={TUNNEL_PROCESS.returncode}'
        remaining = deadline - time.monotonic()
        try:
            response = session.get(
                url + '/_h3_ready',
                headers={'Authorization': 'Bearer ' + API_TOKEN},
                timeout=(min(5, max(0.1, remaining)), min(8, max(0.1, remaining))),
                allow_redirects=False,
            )
            ok, last = _h3_check_response(response)
            if ok:
                return True, last
            if response.status_code == 401:
                return False, last + '；公開端點拒絕 token，請檢查是否連到正確 API'
        except requests.RequestException as exc:
            last = _h3_redact(f'{type(exc).__name__}: {exc}')
        now = time.monotonic()
        if now - last_report >= 15:
            print('公開連線仍在檢查：', last)
            last_report = now
        time.sleep(min(2, max(0, deadline - time.monotonic())))
    return False, last


print('1/3 啟動本機 API……')
# 只停止這份 notebook 建立的 API / tunnel，不停止 ComfyUI 引擎。
_h3_stop_tunnel(globals().get('TUNNEL_PROCESS'))
if globals().get('API_SERVER') is not None:
    API_SERVER.should_exit = True
    old_thread = globals().get('API_THREAD')
    if old_thread is not None:
        old_thread.join(timeout=15)
        if old_thread.is_alive():
            raise RuntimeError('前一次 API 尚未停止，請稍後重跑本格。')

APP = create_app(ENGINE, API_TOKEN)


@APP.get('/_h3_ready')
def _h3_ready():
    # create_app 的全域 Bearer 認證同樣套用於此路由。
    # 這裡不呼叫 /object_info，避免把 ComfyUI 回應速度誤判為 tunnel 失敗。
    return {'protocol': PROTOCOL, 'ready': True}


API_SERVER = uvicorn.Server(uvicorn.Config(
    APP, host=API_HOST, port=API_PORT,
    log_level='warning', access_log=False,
))
API_THREAD = threading.Thread(target=API_SERVER.run, daemon=True)
API_THREAD.start()
for _ in range(150):
    if API_SERVER.started:
        break
    if not API_THREAD.is_alive():
        raise RuntimeError('本機 API 啟動失敗；請查看上方錯誤，確認 7860 埠是否被其他程序占用。')
    time.sleep(0.1)
else:
    raise TimeoutError('本機 API 啟動超過 15 秒。')

# localhost 不經環境代理；公開連線維持 requests 的正常代理設定。
_local_session = requests.Session()
_local_session.trust_env = False
try:
    response = _local_session.get(
        LOCAL_API_URL + '/_h3_ready',
        headers={'Authorization': 'Bearer ' + API_TOKEN}, timeout=(3, 10),
    )
    ok, reason = _h3_check_response(response)
    if not ok:
        raise RuntimeError('本機 API 驗證失敗：' + reason)
    # 真正檢查 API -> ComfyUI；只在建立 tunnel 前做，不放進公開重試迴圈。
    response = _local_session.get(
        LOCAL_API_URL + '/health',
        headers={'Authorization': 'Bearer ' + API_TOKEN}, timeout=(3, 75),
    )
    ok, reason = _h3_check_response(response)
    if not ok:
        raise RuntimeError('本機 /health 失敗，先檢查 Cell 5：' + reason)
except requests.RequestException as exc:
    raise RuntimeError('本機 API / ComfyUI 檢查失敗：' + _h3_redact(exc)) from None
finally:
    _local_session.close()
print('本機 API、token 與 ComfyUI 檢查通過。')

print('2/3 建立 HTTPS tunnel……')
executable = Path('/content/cloudflared')
if not executable.is_file():
    subprocess.run([
        'curl', '-L', '--fail', '--retry', '3',
        '--connect-timeout', '15',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-o', str(executable),
    ], check=True)
executable.chmod(0o755)

with TUNNEL_LOG.open('w', encoding='utf-8') as stream:
    TUNNEL_PROCESS = subprocess.Popen([
        str(executable), 'tunnel', '--no-autoupdate',
        '--protocol', TUNNEL_PROTOCOL,
        '--url', LOCAL_API_URL,
    ], stdout=stream, stderr=subprocess.STDOUT)

H3_API_URL = ''
deadline = time.monotonic() + 90
try:
    while time.monotonic() < deadline:
        if TUNNEL_PROCESS.poll() is not None:
            raise RuntimeError('cloudflared 啟動後退出。')
        log_text = TUNNEL_LOG.read_text(encoding='utf-8', errors='replace')
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
        if match:
            H3_API_URL = match.group(0)
            break
        time.sleep(0.5)
    if not H3_API_URL:
        _h3_stop_tunnel(TUNNEL_PROCESS)
        raise TimeoutError('90 秒內未取得 tunnel 網址。')
except BaseException:
    _h3_stop_tunnel(TUNNEL_PROCESS)
    print(_h3_log_tail())
    raise

print('取得網址：', H3_API_URL)
print('3/3 驗證公開連線；網址剛建立時可能需要等待……')
_public_session = requests.Session()
try:
    PUBLIC_API_VERIFIED, PUBLIC_API_DIAGNOSTIC = _h3_public_check(_public_session, H3_API_URL)
except BaseException:
    print('已停止公開檢查；API / tunnel 保留，可用下方環境變數從本機測試。')
    print(f'$env:H3_API_URL="{H3_API_URL}"')
    raise
finally:
    _public_session.close()

if PUBLIC_API_VERIFIED:
    print('公開 API 驗證成功：', PUBLIC_API_DIAGNOSTIC)
else:
    # 不因 Colab 到公開網址的探測失敗就銷毀 tunnel。
    # 這不代表外部已可用；必須再從使用者本機測試。
    print('公開 API 尚未驗證成功，不能視為已可遠端使用。')
    print('最後原因：', PUBLIC_API_DIAGNOSTIC)
    print('保留 API / tunnel 供本機測試；不需要重新下載模型。')
    print('Tunnel log 尾端：\n' + _h3_log_tail())

print('\nPowerShell 設定：')
print(f'$env:H3_API_URL="{H3_API_URL}"')
print('H3_API_TOKEN 使用和 Colab 相同的一組；本格不顯示 token。')
print('python path/to/minimax-h3-remote-skill/scripts/h3_client.py health')
